# Model Selection & Evaluation: Choosing the Right Model

A systematic guide to selecting, validating, and comparing machine learning models. This notebook covers the theory and practice of model evaluation — from bias-variance tradeoffs to nested cross-validation — ensuring you pick models that **generalize** to unseen data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import FancyBboxPatch

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)

%matplotlib inline

---
## 1. The Goal: Generalization, Not Memorization

A model that **memorizes** training data (low training error) but fails on new data has **overfit**. A model that is too simple to capture patterns **underfits**. The sweet spot is a model that **generalizes** — performing well on both seen and unseen data.

---
## 2. Bias-Variance Tradeoff

- **Bias**: Error from overly simplistic assumptions (underfitting).
- **Variance**: Error from sensitivity to small fluctuations in training data (overfitting).
- **Irreducible Error**: Noise inherent in the problem.

$$\text{Total Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Error}$$

In [ ]:
# Visualization 1: Bias-Variance Tradeoff Diagram

model_complexity = np.linspace(0.1, 1.0, 100)
bias_sq = (1 - model_complexity) ** 2
variance = model_complexity ** 2
irreducible = 0.1 * np.ones_like(model_complexity)
total_error = bias_sq + variance + irreducible

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(model_complexity, bias_sq, label=r'Bias$^2$', lw=3, color='#e74c3c')
ax.plot(model_complexity, variance, label='Variance', lw=3, color='#3498db')
ax.plot(model_complexity, irreducible, label='Irreducible Error', lw=3, color='#95a5a6', ls='--')
ax.plot(model_complexity, total_error, label='Total Error', lw=4, color='#2c3e50')

# Highlight underfitting / overfitting regions
ax.axvspan(0, 0.35, alpha=0.08, color='#e74c3c', label='Underfitting (High Bias)')
ax.axvspan(0.65, 1.0, alpha=0.08, color='#3498db', label='Overfitting (High Variance)')

opt_idx = np.argmin(total_error)
ax.scatter(model_complexity[opt_idx], total_error[opt_idx], color='#27ae60', s=200, zorder=5,
           label='Optimal Complexity')

ax.set_xlabel('Model Complexity', fontsize=13)
ax.set_ylabel('Error', fontsize=13)
ax.set_title('Bias-Variance Tradeoff', fontsize=15, fontweight='bold')
ax.legend(fontsize=11, loc='upper left')
ax.set_ylim(-0.05, 1.5)
ax.set_xlim(-0.02, 1.02)
sns.despine()
plt.tight_layout()
plt.show()

---
## 3. Train / Validation / Test Split — Why Three Sets?

- **Training set**: Fit the model.
- **Validation set**: Tune hyperparameters and make model selection decisions.
- **Test set**: Final, one-time evaluation of the chosen model.

Why not just two? Because using the test set repeatedly for model selection leaks information — you'd overfit to the test set, losing its ability to estimate generalization error. The validation set acts as a **buffer**.

In [ ]:
from sklearn.model_selection import train_test_split

X_full, y_full = np.random.rand(200, 5), np.random.randint(0, 2, 200)

# Step 1: split off a held-out test set
X_temp, X_test, y_temp, y_test = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

# Step 2: split remaining into train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f'Training set:       {X_train.shape[0]:3d} samples')
print(f'Validation set:     {X_val.shape[0]:3d} samples')
print(f'Test set (held-out):{X_test.shape[0]:3d} samples')

---
## 4. Cross-Validation Strategies

Cross-validation (CV) uses multiple train/validation splits to get a **more robust** estimate of model performance than a single split. Below we explore the most common strategies.

In [ ]:
# Visualization 2: CV Fold Visualization

from sklearn.model_selection import (KFold, StratifiedKFold, LeaveOneOut,
                                     RepeatedKFold, ShuffleSplit, GroupKFold)

def plot_cv_folds(cv, name, n_splits=5, n_samples=30, groups=None):
    """Visualise how CV splits the data."""
    fig, ax = plt.subplots(figsize=(12, 4))
    X_cv = np.arange(n_samples).reshape(-1, 1)
    y_cv = np.array([0]*15 + [1]*15)
    if groups is None:
        groups = np.array([0]*10 + [1]*10 + [2]*10)

    for i, (train_idx, test_idx) in enumerate(cv.split(X_cv, y_cv, groups)):
        if i >= n_splits:
            break
        ax.scatter(train_idx, [i+1]*len(train_idx), c='#2ecc71', s=18, marker='s',
                   label='Train' if i == 0 else '')
        ax.scatter(test_idx, [i+1]*len(test_idx), c='#e74c3c', s=18, marker='s',
                   label='Test' if i == 0 else '')

    ax.set_xlabel('Sample index', fontsize=12)
    ax.set_ylabel('Fold', fontsize=12)
    ax.set_title(name, fontsize=14, fontweight='bold')
    ax.set_yticks(range(1, n_splits + 1))
    ax.legend(fontsize=10, loc='upper right')
    ax.set_xlim(-0.5, n_samples - 0.5)
    sns.despine()
    plt.tight_layout()
    plt.show()


# --- K-Fold (k=5) ---
plot_cv_folds(KFold(n_splits=5, shuffle=True, random_state=42),
              'K-Fold (k=5)', n_splits=5)

# --- Stratified K-Fold ---
plot_cv_folds(StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
              'Stratified K-Fold (k=5)')

# --- Leave-One-Out (LOOCV) ---
plot_cv_folds(LeaveOneOut(), 'Leave-One-Out (LOOCV)',
             n_splits=5, n_samples=10)  # only 10 samples for readability

# --- Repeated K-Fold ---
plot_cv_folds(RepeatedKFold(n_splits=5, n_repeats=2, random_state=42),
              'Repeated K-Fold (k=5, 2 repeats)')

# --- ShuffleSplit ---
plot_cv_folds(ShuffleSplit(n_splits=5, test_size=0.3, random_state=42),
              'ShuffleSplit (test=30%)')

# --- Group K-Fold ---
plot_cv_folds(GroupKFold(n_splits=5), 'Group K-Fold (3 groups)')

### When to use which?

| Method | When to use |
|--------|-------------|
| **K-Fold** (k=5 or 10) | Default choice; balanced bias-variance in the estimate |
| **Stratified K-Fold** | Classification with imbalanced classes |
| **LOOCV** | Very small datasets (< 50 samples) |
| **Repeated K-Fold** | Want a more precise estimate (more folds, more computation) |
| **ShuffleSplit** | When you want random permutations, not fixed folds |
| **Group K-Fold** | When samples belong to groups (e.g., same patient, same session) |

---
## 5. Cross-Validation in scikit-learn

Two convenience functions make CV easy:
- `cross_val_score` — returns a single metric per fold
- `cross_validate` — returns multiple metrics, fit times, and optionally the fitted models

In [ ]:
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, cross_validate

X_cv, y_cv = make_classification(n_samples=300, n_features=10, n_informative=6,
                                  n_redundant=2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)

# cross_val_score — single metric
scores = cross_val_score(model, X_cv, y_cv, cv=5, scoring='accuracy')
print(f'Cross-val scores (accuracy): {scores}')
print(f'Mean ± STD:  {scores.mean():.4f} ± {scores.std():.4f}\n')

# cross_validate — multiple metrics
cv_results = cross_validate(
    model, X_cv, y_cv, cv=5,
    scoring={'accuracy': 'accuracy', 'f1': 'f1', 'roc_auc': 'roc_auc'},
    return_train_score=True
)

for metric in ['accuracy', 'f1', 'roc_auc']:
    train_scores = cv_results[f'train_{metric}']
    test_scores = cv_results[f'test_{metric}']
    print(f'{metric:15s}  Train: {train_scores.mean():.4f} ± {train_scores.std():.4f}  '
          f'Test:  {test_scores.mean():.4f} ± {test_scores.std():.4f}')

---
## 6. Learning Curves: Diagnose Bias vs Variance

A learning curve plots training & validation error against **training set size**. It reveals:
- **High Bias**: Both curves converge to a high error; more data won't help much.
- **High Variance**: Large gap between curves; more data might help.

In [ ]:
# Visualization 3: Learning Curve

from sklearn.model_selection import LearningCurveDisplay
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

X_lc, y_lc = make_classification(n_samples=500, n_features=5, n_informative=4,
                                  n_redundant=0, random_state=1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# High Bias: shallow tree (max_depth=2)
LearningCurveDisplay.from_estimator(
    DecisionTreeClassifier(max_depth=2, random_state=42),
    X_lc, y_lc, cv=5, scoring='accuracy', n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10),
    ax=axes[0]
)
axes[0].set_title('High Bias (max_depth=2)', fontsize=13, fontweight='bold')

# High Variance: deep tree (max_depth=None)
LearningCurveDisplay.from_estimator(
    DecisionTreeClassifier(max_depth=None, random_state=42),
    X_lc, y_lc, cv=5, scoring='accuracy', n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10),
    ax=axes[1]
)
axes[1].set_title('High Variance (max_depth=None)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 7. Validation Curves: Tuning Hyperparameters

A validation curve plots training & validation scores against a **hyperparameter** value. The optimal value is where validation score peaks (and the gap to training score is reasonable).

In [ ]:
# Visualization 4: Validation Curve

from sklearn.model_selection import ValidationCurveDisplay

fig, ax = plt.subplots(figsize=(10, 5))

ValidationCurveDisplay.from_estimator(
    RandomForestClassifier(random_state=42),
    X_lc, y_lc,
    param_name='max_depth',
    param_range=np.arange(1, 21),
    cv=5, scoring='accuracy', n_jobs=-1,
    ax=ax
)
ax.set_title('Validation Curve: Tuning max_depth', fontsize=14, fontweight='bold')
ax.axvline(x=6, color='#27ae60', ls='--', lw=2, label='Recommended ~6')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
## 8. Grid Search: Exhaustive Hyperparameter Search

`GridSearchCV` tries every combination of hyperparameters in a grid and picks the best via CV.

In [ ]:
from sklearn.model_selection import GridSearchCV

X_gs, y_gs = make_classification(n_samples=300, n_features=8, random_state=42)

param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, 7, None],
    'min_samples_split': [2, 5, 10]
}

gs = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=0
)
gs.fit(X_gs, y_gs)

print(f'Best parameters: {gs.best_params_}')
print(f'Best CV accuracy: {gs.best_score_:.4f}')
print(f'Grid searched {gs.n_splits_ * len(param_grid["n_estimators"]) * len(param_grid["max_depth"]) * len(param_grid["min_samples_split"])} combinations')

---
## 9. Randomized Search: Faster Alternative

`RandomizedSearchCV` samples random combinations from a distribution — much faster when the search space is large. Often finds equally good (or better) hyperparameters in fewer iterations.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_dist = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(3, 20),
    'min_samples_split': randint(2, 20),
    'max_features': uniform(0.5, 0.5)  # range 0.5-1.0
}

rs = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_dist, n_iter=30, cv=5, scoring='accuracy',
    n_jobs=-1, random_state=42
)
rs.fit(X_gs, y_gs)

print(f'Best parameters (RS): {rs.best_params_}')
print(f'Best CV accuracy (RS): {rs.best_score_:.4f}')

---
## 10. Halving Grid/Random Search (Successive Halving)

These use **successive halving**: start with many candidates on a small budget, then iteratively淘汰 the bottom half and increase the budget for the survivors. Much more efficient than standard grid/random search.

In [ ]:
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV, HalvingRandomSearchCV

# Halving Grid Search
param_grid_small = {
    'max_depth': [3, 5, 7, 10, 15, None],
    'min_samples_split': [2, 5, 10]
}

hgs = HalvingGridSearchCV(
    RandomForestClassifier(n_estimators=100, random_state=42),
    param_grid_small, cv=3, factor=2, scoring='accuracy',
    n_jobs=-1, verbose=0
)
hgs.fit(X_gs, y_gs)

print(f'HalvingGrid best params: {hgs.best_params_}')
print(f'HalvingGrid best score:  {hgs.best_score_:.4f}')
print(f'Nº candidates evaluated:  {len(hgs.cv_results_["params"])}')

# Halving Random Search
hrs = HalvingRandomSearchCV(
    RandomForestClassifier(n_estimators=100, random_state=42),
    param_dist, n_candidates='exhaust', factor=2, cv=3,
    scoring='accuracy', n_jobs=-1, random_state=42
)
hrs.fit(X_gs, y_gs)

print(f'HalvingRandom best params: {hrs.best_params_}')
print(f'HalvingRandom best score:  {hrs.best_score_:.4f}')

---
## 11. Evaluation Metrics Deep Dive

Choosing the **right metric** is as important as choosing the right model.

### Classification Metrics

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, log_loss,
                             matthews_corrcoef, confusion_matrix)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import label_binarize

# Generate imbalanced data
X_imb, y_imb = make_classification(
    n_samples=1000, n_features=5,
    weights=[0.9, 0.1],  # 90% class 0, 10% class 1
    flip_y=0.05, random_state=42
)

# Dummy classifier that predicts majority class
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy='most_frequent').fit(X_imb, y_imb)
y_pred_dummy = dummy.predict(X_imb)
y_prob_dummy = dummy.predict_proba(X_imb)[:, 1]

# Real model
lr = LogisticRegression().fit(X_imb, y_imb)
y_pred_lr = lr.predict(X_imb)
y_prob_lr = lr.predict_proba(X_imb)[:, 1]

print(f"{'Metric':20s} {'Dummy':>10s} {'Logistic':>10s}")
print('-'*42)
print(f"{'Accuracy':20s} {accuracy_score(y_imb, y_pred_dummy):10.3f} {accuracy_score(y_imb, y_pred_lr):10.3f}")
print(f"{'Precision':20s} {precision_score(y_imb, y_pred_dummy):10.3f} {precision_score(y_imb, y_pred_lr):10.3f}")
print(f"{'Recall':20s} {recall_score(y_imb, y_pred_dummy):10.3f} {recall_score(y_imb, y_pred_lr):10.3f}")
print(f"{'F1-score':20s} {f1_score(y_imb, y_pred_dummy):10.3f} {f1_score(y_imb, y_pred_lr):10.3f}")
print(f"{'ROC-AUC':20s} {roc_auc_score(y_imb, y_prob_dummy):10.3f} {roc_auc_score(y_imb, y_prob_lr):10.3f}")
print(f"{'Log Loss':20s} {log_loss(y_imb, y_prob_dummy):10.3f} {log_loss(y_imb, y_prob_lr):10.3f}")
print(f"{'Matthews Corr':20s} {matthews_corrcoef(y_imb, y_pred_dummy):10.3f} {matthews_corrcoef(y_imb, y_pred_lr):10.3f}")

print(f"\n→ Dummy 'accuracy' is {accuracy_score(y_imb, y_pred_dummy):.3f} but F1 is {f1_score(y_imb, y_pred_dummy):.3f}! "
      "Accuracy is misleading on imbalanced data.")

### When Accuracy is Misleading

With a 90%/10% class imbalance, a dummy that always predicts the majority class gets **90% accuracy** but **0% recall** on the minority class. This is why you must look beyond accuracy for imbalanced problems.

### Regression Metrics

In [ ]:
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             root_mean_squared_error, r2_score,
                             mean_absolute_percentage_error)
from sklearn.datasets import make_regression
from sklearn.ensemble import GradientBoostingRegressor

X_reg, y_reg = make_regression(n_samples=200, n_features=5, noise=10, random_state=42)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

gbr = GradientBoostingRegressor(random_state=42).fit(X_train_r, y_train_r)
y_pred = gbr.predict(X_test_r)

n = len(y_test_r)
p = X_test_r.shape[1]
r2 = r2_score(y_test_r, y_pred)
adj_r2 = 1 - ((1 - r2) * (n - 1) / (n - p - 1))

print(f'MAE : {mean_absolute_error(y_test_r, y_pred):.3f}')
print(f'MSE : {mean_squared_error(y_test_r, y_pred):.3f}')
print(f'RMSE: {root_mean_squared_error(y_test_r, y_pred):.3f}')
print(f'MAPE: {mean_absolute_percentage_error(y_test_r, y_pred):.3f}')
print(f'R²  : {r2:.3f}')
print(f'Adj R²: {adj_r2:.3f}')

---
## 12. Multi-Class Metrics: Macro, Micro, Weighted

For multi-class problems, metrics like precision/recall/F1 can be averaged across classes:

- **Macro**: unweighted mean per class (treats all classes equally).
- **Micro**: global count of TP/FP/FN (favors majority class).
- **Weighted**: mean weighted by class support (recommended for imbalanced).

In [ ]:
from sklearn.datasets import make_classification
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC

X_mc, y_mc = make_classification(
    n_samples=200, n_features=5, n_classes=3, n_informative=3,
    weights=[0.5, 0.3, 0.2], random_state=42
)
X_train_mc, X_test_mc, y_train_mc, y_test_mc = train_test_split(
    X_mc, y_mc, test_size=0.3, random_state=42, stratify=y_mc
)

svm = OneVsRestClassifier(SVC(kernel='linear', probability=True, random_state=42))
svm.fit(X_train_mc, y_train_mc)
y_pred_mc = svm.predict(X_test_mc)

print(f"{'Average':15s} {'Precision':10s} {'Recall':10s} {'F1':10s}")
print('-'*47)
for avg in ['macro', 'micro', 'weighted']:
    prec = precision_score(y_test_mc, y_pred_mc, average=avg)
    rec = recall_score(y_test_mc, y_pred_mc, average=avg)
    f1 = f1_score(y_test_mc, y_pred_mc, average=avg)
    print(f'{avg:15s} {prec:.4f}      {rec:.4f}      {f1:.4f}')

---
## 13. Probability Calibration & Reliability Diagrams

A model is **well-calibrated** if predicted probabilities match empirical frequencies (e.g., among samples with ~70% predicted probability, ~70% are actually positive). A **reliability diagram** (calibration curve) plots this visually.

In [ ]:
# Visualization 5: Calibration Curve

from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.naive_bayes import GaussianNB

X_cal, y_cal = make_classification(n_samples=1000, n_features=10, random_state=42)
X_cal_tr, X_cal_te, y_cal_tr, y_cal_te = train_test_split(
    X_cal, y_cal, test_size=0.3, random_state=42
)

# Uncalibrated
nb = GaussianNB().fit(X_cal_tr, y_cal_tr)
prob_pos_nb = nb.predict_proba(X_cal_te)[:, 1]

# Calibrated (Platt scaling)
nb_cal = CalibratedClassifierCV(GaussianNB(), cv=3, method='sigmoid')
nb_cal.fit(X_cal_tr, y_cal_tr)
prob_pos_cal = nb_cal.predict_proba(X_cal_te)[:, 1]

fig, ax = plt.subplots(figsize=(8, 7))

for prob, label, marker, color in [
    (prob_pos_nb, 'Naive Bayes (uncalibrated)', 'o', '#e74c3c'),
    (prob_pos_cal, 'Naive Bayes (calibrated)', 's', '#3498db'),
]:
    fraction_pos, mean_pred = calibration_curve(y_cal_te, prob, n_bins=10)
    ax.plot(mean_pred, fraction_pos, marker=marker, lw=2, label=label, color=color)

ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Perfectly calibrated')
ax.set_xlabel('Mean predicted probability', fontsize=12)
ax.set_ylabel('Fraction of positives', fontsize=12)
ax.set_title('Calibration (Reliability) Diagram', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
sns.despine()
plt.tight_layout()
plt.show()

---
## 14. ROC Curves: Comparing Classifiers

The ROC curve plots TPR vs FPR at various thresholds. AUC (area under the curve) summarizes overall discriminative ability. Below we compare multiple classifiers.

In [ ]:
# Visualization 6: ROC Curves Comparison

from sklearn.metrics import RocCurveDisplay
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

X_roc, y_roc = make_classification(n_samples=500, n_features=10, random_state=42)
X_roc_tr, X_roc_te, y_roc_tr, y_roc_te = train_test_split(
    X_roc, y_roc, test_size=0.3, random_state=42
)

classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree (d=3)': DecisionTreeClassifier(max_depth=3),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5)
}

fig, ax = plt.subplots(figsize=(9, 7))

for name, clf in classifiers.items():
    clf.fit(X_roc_tr, y_roc_tr)
    RocCurveDisplay.from_estimator(clf, X_roc_te, y_roc_te, ax=ax, name=name, lw=2)

ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Random (AUC=0.5)')
ax.set_title('ROC Curves — Classifier Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
sns.despine()
plt.tight_layout()
plt.show()

---
## 15. Comparing Models Statistically

We can use a **paired t-test** on the per-fold CV scores of two models to test if one is significantly better than the other.

In [ ]:
from scipy.stats import ttest_rel

X_tt, y_tt = make_classification(n_samples=300, n_features=10, random_state=42)

model_a = LogisticRegression(max_iter=1000)
model_b = DecisionTreeClassifier(max_depth=3, random_state=42)

scores_a = cross_val_score(model_a, X_tt, y_tt, cv=10, scoring='accuracy')
scores_b = cross_val_score(model_b, X_tt, y_tt, cv=10, scoring='accuracy')

stat, p_value = ttest_rel(scores_a, scores_b)

print(f'Logistic Regression: {scores_a.mean():.4f} ± {scores_a.std():.4f}')
print(f'Decision Tree (d=3): {scores_b.mean():.4f} ± {scores_b.std():.4f}')
print(f'Paired t-test: t = {stat:.3f}, p = {p_value:.4f}')
print(f'→ {"Significant difference" if p_value < 0.05 else "No significant difference"} (α=0.05)')

---
## 16. Nested Cross-Validation for Unbiased Model Comparison

When you use CV results to **select a model or tune hyperparameters**, the CV scores become optimistically biased. **Nested CV** separates model selection (inner loop) from performance estimation (outer loop) for an unbiased evaluation.

- **Outer loop** (e.g., 5-fold): estimates generalization error.
- **Inner loop** (e.g., 3-fold): does hyperparameter tuning on each outer training fold.

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score, KFold

X_nest, y_nest = make_classification(n_samples=200, n_features=8, random_state=42)

# Inner CV: tune hyperparameters
param_grid = {'C': [0.01, 0.1, 1, 10]}
inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

clf = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid, cv=inner_cv, scoring='accuracy'
)

# Outer CV: unbiased performance estimate
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
nested_scores = cross_val_score(clf, X_nest, y_nest, cv=outer_cv, scoring='accuracy')

print(f'Nested CV scores: {nested_scores}')
print(f'Nested CV accuracy: {nested_scores.mean():.4f} ± {nested_scores.std():.4f}')

---
## 17. The No Free Lunch Theorem

> "No single algorithm is universally the best for all problems."

Averaged over all possible data distributions, every algorithm has the same expected performance. This means **model selection must be problem-specific** — guided by data exploration, domain knowledge, and empirical evaluation.

**Practical takeaways:**
- Always try multiple families of models.
- Use CV to evaluate each one.
- Let the data (and domain) guide your choice — not dogma.

---
## Practice Exercises

### Exercise 1: Bias-Variance Intuition

Train a `LinearRegression`, a `DecisionTreeRegressor(max_depth=3)`, and a `DecisionTreeRegressor(max_depth=None)` on the same regression dataset. Plot their learning curves. Which model has highest bias? Which has highest variance?

```python
# Your code here
```

### Exercise 2: K-Fold vs Stratified K-Fold

Create an imbalanced dataset (95% / 5% split). Compute `cross_val_score` with `KFold` and `StratifiedKFold` (k=5). Compare the mean and std of the scores. Why might one give more stable results?

```python
# Your code here
```

### Exercise 3: Grid Search + Validation Curve

For a `RandomForestClassifier`, use `GridSearchCV` to tune `n_estimators` and `max_features`. Then plot a validation curve for `n_estimators` with `max_features` fixed to the best value. Compare the validation curve to the grid search results.

```python
# Your code here
```

### Exercise 4: Metric Selection for Imbalanced Data

Generate a dataset with 2% positive class. Train three classifiers (Dummy, LogisticRegression, RandomForest). Report accuracy, precision, recall, F1, ROC-AUC, and MCC. Which metrics correctly identify the RandomForest as best?

```python
# Your code here
```

### Exercise 5: Nested CV Comparison

Compare `SVC(kernel='rbf')` and `RandomForestClassifier` using nested CV. In the inner loop, tune `C`/`gamma` for SVC and `n_estimators`/`max_depth` for RF. Which model generalizes better?

```python
# Your code here
```